# COMPACT-MI-RFE — Colab Pro
Stage 6 setup. Chạy tuần tự 12 cell. Mặc định chỉ dry run; hai kết quả RF FS16 không chạy lại. Không dùng Test trước khi khóa lựa chọn và refit.


In [ ]:
# CELL 1 — Runtime information (read-only; no GPU acceleration assumed)
import os, platform, shutil, sys, importlib.metadata as im
from pathlib import Path
try:
    import psutil
    vm = psutil.virtual_memory()
    ram = {"total": vm.total, "available": vm.available}
    physical = psutil.cpu_count(logical=False)
except ImportError:
    ram, physical = {"total": None, "available": None}, None
disk = shutil.disk_usage("/content")
versions = {}
for name in ("numpy","pandas","scipy","scikit-learn","joblib","psutil"):
    try: versions[name] = im.version(name)
    except im.PackageNotFoundError: versions[name] = None
gpu_tool = shutil.which("nvidia-smi")
print({"os":platform.platform(),"arch":platform.machine(),"python":sys.version,
       "logical_cpus":os.cpu_count(),"physical_cpus":physical,"ram_bytes":ram,
       "disk_free_bytes":disk.free,"gpu_probe_available":bool(gpu_tool),"versions":versions})
print("This scikit-learn pipeline runs on CPU; Colab Pro/High-RAM is not assumed.")

In [ ]:
# CELL 2 — Mount Google Drive (you authorize the Google prompt)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted:", Path('/content/drive/MyDrive').is_dir())

In [ ]:
# CELL 3 — Locate dataset source without assuming Shared-with-me is in MyDrive
SHARED_FOLDER_ID = "1Ixy5iJdeavkEbFJ3cy6LjkQ8h300OAxh"
DRIVE_DATASET_PATH = ""  #@param {type:"string"}
DRIVE_ARCHIVE_ID = ""  #@param {type:"string"}
# Optional: mounted WINDOWS_CPLUS folder, its parent, or a ZIP. Leave blank for shared folder ID.
# The shared folder already contains WINDOWS_CPLUS and CSV_SPLIT_CPLUS; no ZIP is needed.
if DRIVE_DATASET_PATH:
    candidate = Path(DRIVE_DATASET_PATH)
    print({"path":str(candidate),"exists":candidate.exists(),
           "bytes":candidate.stat().st_size if candidate.is_file() else None})
    if not candidate.exists(): raise FileNotFoundError(candidate)
else:
    print("Shared folder ID:",SHARED_FOLDER_ID,"— API discovery deferred to CELL 6 after dependencies.")

In [ ]:
# CELL 4 — Clone safely; existing checkout is inspected, never reset/pulled
import subprocess
REPO = Path('/content/RFforNetwork')
URL = 'https://github.com/kasum00/RFforNetwork.git'
if REPO.exists():
    if not (REPO/'.git').is_dir(): raise FileExistsError(f"Not a Git checkout: {REPO}")
    status = subprocess.run(['git','-C',str(REPO),'status','--porcelain'],
                            text=True,capture_output=True,check=True)
    print("Existing checkout; no pull. Local changes:",status.stdout.splitlines())
else:
    subprocess.run(['git','clone','--',URL,str(REPO)],check=True)
required = ['huyen_code/ciciomt24_training.py','huyen_code/ciciomt24_sliding_window.py',
            'huyen_code/ciciomt24_split.py','huyen_code/test_ciciomt24_training.py',
            'huyen_code/test_ciciomt24_split.py','huyen_code/test_ciciomt24_sliding_window.py',
            'colab/experiment_launcher.py','configs/cplus_manifest/split_manifest.csv',
            'CODE_ctu13/[[CTU13]9_ex2_rf.py']
missing = [p for p in required if not (REPO/p).is_file()]
if missing:
    raise RuntimeError(f"GitHub checkout lacks Stage 6/Stage 5 files: {missing}. Publish selected source files manually; do not use an incomplete clone.")
print("Repository ready:",REPO)

In [ ]:
# CELL 5 — Select the Python-specific scientific profile; install, restart if needed, test
import importlib.metadata as im, subprocess, sys
sys.path.insert(0,str(REPO))
from colab.setup_colab import selected_versions, dependency_plan, repo_inventory
print('Actual Python:',sys.version)
expected = selected_versions()  # Python 3.13 uses NumPy 2.2.6, not 1.26.4
print('Selected scientific profile:',expected)
current = {}
for name in expected:
    try: current[name] = im.version(name)
    except im.PackageNotFoundError: current[name] = None
extras_missing = []
for name in ('psutil','google-api-python-client','google-auth'):
    try: im.version(name)
    except im.PackageNotFoundError: extras_missing.append(name)
science_needs = any(current[name] != version for name,version in expected.items())
needs = science_needs or bool(extras_missing)
if needs:
    subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-colab.txt')],check=True)
    if science_needs:
        raise RuntimeError('Scientific pins installed. Runtime > Restart runtime, then rerun Cells 1–5. No training started.')
subprocess.run([sys.executable,'-m','pip','check'],check=True)
plan = dependency_plan()
if plan['install_needed']: raise RuntimeError(f"Scientific versions still differ: {plan}")
print('Stage 5 exact scientific versions:',plan['stage5_exact_versions'])
print('Python 3.13 NumPy 2.x may change numerical results; Stage 5 bitwise parity is not claimed.')
if not repo_inventory(REPO)['pass']: raise RuntimeError(repo_inventory(REPO))
tests = subprocess.run([sys.executable,'-m','unittest','-v',
                        'test_ciciomt24_training','test_ciciomt24_split',
                        'test_ciciomt24_sliding_window'],
                       cwd=REPO/'huyen_code',capture_output=True,text=True)
print(tests.stdout[-3000:]); print(tests.stderr[-6000:])
if tests.returncode: raise RuntimeError("Unit tests failed; full training blocked.")
stage6_tests = subprocess.run([sys.executable,'-m','unittest','-v','colab.test_stage6_colab'],
                              cwd=REPO,capture_output=True,text=True)
print(stage6_tests.stdout[-2000:]); print(stage6_tests.stderr[-5000:])
if stage6_tests.returncode: raise RuntimeError('Stage 6 adapter tests failed; full training blocked.')
from colab.dataset_validator import sha256
UNIT_TEST_EVIDENCE = {"passed":True,
    "training_sha256":sha256(REPO/'huyen_code'/'ciciomt24_training.py')}
print("Unit tests PASS; training source hash:",UNIT_TEST_EVIDENCE['training_sha256'])

In [ ]:
# CELL 6 — Prefer the extracted WINDOWS_CPLUS folder; ZIP is optional
import shutil
from colab.drive_manager import discover_shared_dataset, download_archive
from colab.dataset_validator import (WINDOW_ROOT, inspect_archive, prepare_dataset,
    prepare_drive_folder_dataset, prepare_mounted_folder_dataset)
DATASET_READY = False
if WINDOW_ROOT.exists():
    PREPARED = prepare_dataset(REPO, output_root=WINDOW_ROOT)
elif DRIVE_DATASET_PATH:
    source = Path(DRIVE_DATASET_PATH)
    if source.is_dir():
        windows_source = source/'WINDOWS_CPLUS' if (source/'WINDOWS_CPLUS').is_dir() else source
        PREPARED = prepare_mounted_folder_dataset(REPO,windows_source,output_root=WINDOW_ROOT)
    elif source.is_file():
        print('Optional ZIP inventory:',inspect_archive(source))
        PREPARED = prepare_dataset(REPO,archive=source,output_root=WINDOW_ROOT)
    else:
        raise FileNotFoundError(source)
else:
    found = discover_shared_dataset(SHARED_FOLDER_ID)  # user-authorized read-only API
    print('Shared entries:',found['children'])
    print('Audit folder present (not downloaded):',found['audit_folder_present'])
    if found['mode'] == 'folder':
        PREPARED = prepare_drive_folder_dataset(REPO,found['windows_folder']['id'],
                                                found['service'],output_root=WINDOW_ROOT)
    elif found['mode'] == 'zip_available':
        zips = found['zips']
        chosen = next((x for x in zips if x['id']==DRIVE_ARCHIVE_ID),None) if DRIVE_ARCHIVE_ID else (zips[0] if len(zips)==1 else None)
        if chosen is None: raise RuntimeError('Multiple ZIPs: specify DRIVE_ARCHIVE_ID; no folder found.')
        if (Path(chosen['name']).name != chosen['name'] or chr(92) in chosen['name']
                or chosen['name'] in ('.','..')): raise ValueError('Unsafe Drive ZIP filename')
        size = int(chosen.get('size') or 0)
        free = shutil.disk_usage('/content').free
        if size <= 0 or free < 2*size + 8*1024**3:
            raise OSError(f'ZIP disk admission FAIL: need {2*size+8*1024**3:,} bytes; have {free:,}')
        local_zip = Path('/content/datasets/.archive_downloads')/chosen['name']
        if local_zip.exists():
            if local_zip.stat().st_size != size: raise IOError('Existing ZIP differs; inspect manually.')
        else: download_archive(chosen['id'],local_zip)
        PREPARED = prepare_dataset(REPO,archive=local_zip,output_root=WINDOW_ROOT)
    else:
        raise FileNotFoundError('Shared folder has no WINDOWS_CPLUS folder or ZIP')
DATASET_READY = PREPARED['status'] == 'PASS_METADATA_SCHEMA'
print('Dataset path:',WINDOW_ROOT)
print('Configuration folders:',PREPARED['config_directories'],'CSV files:',PREPARED['csv_files'])
print('Manifest counts:',PREPARED['counts'],'Missing files:',PREPARED['missing_files'])
print('Schema validation:',PREPARED['status'],'DATASET_READY=',DATASET_READY)
print('Drive source was not moved or deleted; Test feature rows were not opened.')

In [ ]:
# CELL 7 — Validate approved C+ counts/schema and proxy labels (Test rows unopened)
from colab.dataset_validator import EXPECTED, validate_dataset
DATA_AUDIT = validate_dataset(WINDOW_ROOT,REPO)
DATASET_READY = DATASET_READY and DATA_AUDIT['status'] == 'PASS_METADATA_SCHEMA'
print("STATUS:",DATA_AUDIT['status'])
for config, counts in DATA_AUDIT['counts'].items():
    key = 'FW'+config.split('_')[1]+'_FS'+config.split('_')[3]
    print(config,counts,DATA_AUDIT['proxy_window_counts'][key])
print("Features:",DATA_AUDIT['features'],"metadata+label:",DATA_AUDIT['metadata_and_label_columns'])
print("Label origin:",DATA_AUDIT['label_origin'])
print("Test feature rows read:",DATA_AUDIT['test_feature_rows_read'])
print('DATASET_READY=',DATASET_READY,'Dataset path:',DATA_AUDIT['dataset_path'])
print("Stage 4.2 readback reused; this metadata/header check is not a fresh full CSV content audit.")

In [ ]:
# CELL 8 — Select exactly one experiment/config; default is DRY RUN
EXPERIMENT = "mi_rfe_rf"  #@param ["mi_rfe_rf", "baseline_svm", "mi_rfe_svm"]
FW = 512  #@param [256, 512]
FS = 8  #@param [8, 16]
RUN_NAME = "mi_rfe_rf_FW512_FS8_run01"  #@param {type:"string"}
MODE = "dry_run"  #@param ["dry_run", "full_train"]
CONFIRMATION = ""  #@param {type:"string"}
RUN_TECHNICAL_SVM_PROBE = False  #@param {type:"boolean"}
from colab.experiment_launcher import RunSpec
from colab.drive_manager import DRIVE_ROOT
SPEC = RunSpec(EXPERIMENT,f"FW_{FW}_FS_{FS}",RUN_NAME)
SPEC.validate()
LOCAL_RUNS = Path('/content/stage6_runs')
print({"experiment":SPEC.experiment,"config":SPEC.config,"run":SPEC.run_name,
       "mode":MODE,"local_output":str(LOCAL_RUNS),"drive_output":str(DRIVE_ROOT)})
print("Full train requires CELL 10 and the exact confirmation phrase.")

In [ ]:
# CELL 9 — Resource/preflight only: NEVER calls model.fit or reads Test feature rows
from colab.experiment_launcher import preflight
PROBE_PATH = LOCAL_RUNS/(SPEC.run_name+'_svm_technical_probe.json')
if not DATASET_READY: raise RuntimeError('Dataset preparation/validation has not passed')
PREFLIGHT = preflight(REPO,WINDOW_ROOT,LOCAL_RUNS,DRIVE_ROOT,SPEC,
                      unit_test_evidence=UNIT_TEST_EVIDENCE,
                      svm_probe_path=PROBE_PATH if PROBE_PATH.exists() else None)
print("PREFLIGHT:",PREFLIGHT['status'])
print("CHECKS:",PREFLIGHT['checks'])
print("RAM GUARD:",PREFLIGHT['ram_guard'])
print("LOCAL DISK REQUIRED BYTES:",PREFLIGHT['local_disk_required_bytes'])
print("Test feature rows read:",PREFLIGHT['test_feature_rows_read'],
      "model.fit calls:",PREFLIGHT['model_fit_calls'])

In [ ]:
# CELL 10 — Optional bounded SVM probe or one explicitly confirmed full run
from colab.experiment_launcher import run_one, run_technical_svm_probe
RUN_RESULT = None
if RUN_TECHNICAL_SVM_PROBE:
    if EXPERIMENT == 'mi_rfe_rf': raise ValueError("SVM probe selected for RF")
    LOCAL_RUNS.mkdir(parents=True,exist_ok=True)
    probe = run_technical_svm_probe(REPO,WINDOW_ROOT,SPEC,PROBE_PATH)
    print("TECHNICAL-ONLY probe:",probe['status'],"Test CSV rows:",probe['test_csv_rows_read'])
    PREFLIGHT = preflight(REPO,WINDOW_ROOT,LOCAL_RUNS,DRIVE_ROOT,SPEC,
                          unit_test_evidence=UNIT_TEST_EVIDENCE,svm_probe_path=PROBE_PATH)
    print("Updated preflight:",PREFLIGHT['status'],PREFLIGHT['checks'])
    if MODE == 'full_train': raise RuntimeError('SVM probe finished. Review its resource/timing record, turn probe toggle off, rerun CELL 9, then explicitly start full train.')
if MODE == 'full_train':
    if PREFLIGHT['status'] != 'PASS': raise RuntimeError("Preflight FAIL: no training started.")
    RUN_RESULT = run_one(REPO,WINDOW_ROOT,LOCAL_RUNS,DRIVE_ROOT,SPEC,
                         unit_test_evidence=UNIT_TEST_EVIDENCE,
                         confirmation=CONFIRMATION,
                         svm_probe_path=PROBE_PATH if PROBE_PATH.exists() else None)
    print("OFFICIAL RUN:",RUN_RESULT)
else:
    print("Dry run only. No full model fit, no Test evaluation.")
    print("To run one official experiment, select full_train, type:",
          f"RUN {SPEC.experiment} {SPEC.config} {SPEC.run_name}",
          "and rerun CELL 9 then CELL 10.")

In [ ]:
# CELL 11 — Verify publication; optionally promote an ALREADY COMPLETED checkpoint
import json
RECOVER_COMPLETED_CHECKPOINT = False  #@param {type:"boolean"}
if RECOVER_COMPLETED_CHECKPOINT:
    from colab.checkpoint_manager import publish_completed_checkpoint
    from colab.dataset_validator import EXPECTED
    recovered_path = publish_completed_checkpoint(DRIVE_ROOT,SPEC.run_name,EXPECTED[SPEC.config])
    print('Promoted existing completed checkpoint without fit or Test evaluation:',recovered_path)
if RUN_RESULT is None:
    print("No new official result to save; no Drive result folder created by this cell.")
else:
    result_dir = Path(RUN_RESULT['drive_result'])
    manifest = json.loads((result_dir/'artifact_manifest.json').read_text())
    for name, expected in manifest['files'].items():
        path = result_dir/name
        if path.stat().st_size != expected['bytes'] or sha256(path) != expected['sha256']:
            raise IOError(f"Drive result artifact failed verification: {name}")
    print("Drive artifacts verified:",result_dir)
    print("Environment:",DRIVE_ROOT/'environment'/(SPEC.run_name+'.json'))
    print("Checkpoints:",DRIVE_ROOT/'checkpoints'/(SPEC.run_name+'.incomplete'))
    print("No Test prediction/evaluation was repeated by this cell.")

In [ ]:
# CELL 12 — Read result.json only; never load Test CSV/model for summary
from colab.results_summary import completed_results, previous_artifacts_status
rows = completed_results(DRIVE_ROOT)
print("Previous FS16 artifacts:",previous_artifacts_status(DRIVE_ROOT))
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows: print(row)
print("SCENARIO_PROXY; Benign is rare; capture/session independence remains unproven.")